In [1]:
import json
import pandas as pd
import re
import os
import datasets

/projects/iris/rferreir/.envs/spe/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Path to the directory containing the GLOBAL data
global_path = "/projects/iris/rferreir/datasets/SpatialEvalLLM/map_global"

# Path to the directory containing the LOCAL data
local_path = "/projects/iris/rferreir/datasets/SpatialEvalLLM/map_local"

# Path to the directory where the files will be saved
output_dir = '/projects/iris/rferreir/GeoBenchmark/data/SpatialEvalLLM'

In [15]:
def get_data(dir_path, description_level='global'):
    final_data = []
    for filename in os.listdir(dir_path):
        if filename.endswith(".jsonl"): 
            path = os.path.join(dir_path, filename)
            print(path)
            with open(path, 'r') as f:
                for line in f:
                    line = line.strip()
                    if line:  # ignore empty lines
                        data = json.loads(line)
                        data = {key:data[key] for key in ['question', 'answer']}
                        split = data['question'].split('.')
                        data['scenario'] = ".".join(split[:-1]) + "."
                        data['question'] = split[-1].strip()
                        if '?' not in data['question']:
                            print(data['question'])
                        data['answer'] = data['answer'].replace('.', '') #get rid of useless punctuation
                        matches = re.findall(r"(\w+)-([^\_]+)", filename.split('.')[0])
                        info = {key.replace('_', ''): value for key, value in matches}
                        data['struct_type'] = info['type']
                        data['size'] = info['size']
                        steps = 'steps' if 'steps' in info.keys() else 'nstep'
                        data['k_hop'] = info[steps]
                        data['seed'] = info['seed']
                        data['description_level'] = description_level
                        final_data.append(data)
    return final_data

## Global data

In [16]:
final_data = get_data(global_path)
print(len(final_data))

/projects/iris/rferreir/datasets/SpatialEvalLLM/map_global/type-square_size-3_steps-8_seed-3_n-100_special_order-snake_order.jsonl
/projects/iris/rferreir/datasets/SpatialEvalLLM/map_global/type-ring_size-12_steps-8_seed-12_n-100.jsonl
/projects/iris/rferreir/datasets/SpatialEvalLLM/map_global/type-ring_size-9_steps-4_seed-9_n-100.jsonl
/projects/iris/rferreir/datasets/SpatialEvalLLM/map_global/type-square_size-3_steps-8_seed-3_n-100_special_order-random_order.jsonl
/projects/iris/rferreir/datasets/SpatialEvalLLM/map_global/type-square_size-3_steps-4_seed-3_n-100.jsonl
/projects/iris/rferreir/datasets/SpatialEvalLLM/map_global/type-square_size-3_steps-8_seed-3_n-100.jsonl
/projects/iris/rferreir/datasets/SpatialEvalLLM/map_global/type-square_size-3_steps-8_seed-3_n-100_special_order-snake_order_with_coordinates.jsonl
/projects/iris/rferreir/datasets/SpatialEvalLLM/map_global/type-tree_size-9_steps-4_seed-9_n-100.jsonl
/projects/iris/rferreir/datasets/SpatialEvalLLM/map_global/type-tree

In [17]:
print(final_data[0])

{'question': 'What will you find?', 'answer': 'half-track', 'scenario': 'You have been given a 3 by 3 square grid. Starting from a vertex, you will move along the edges of the grid. Initially, you are positioned at the bottom-left corner of the grid, where you will find a marimba, then you go right, where you will find a Scottish Terrier, then you go right, where you will find a military uniform. Then you go up, where you will find a radio telescope, then you go left, where you will find a half-track, then you go left, where you will find a restaurant. Then you go up, where you will find a wok, then you go right, where you will find a sleeping bag, then you go right, where you will find a balance beam. Now you have all the information on the map. You start at the position where the half-track is located, then you go left by one step, then you go up by one step, then you go right by one step, then you go right by one step, then you go down by one step, then you go down by one step, then

In [18]:
with open(os.path.join(output_dir, 'data_global.json'), 'w') as f:
    json.dump(final_data, f)

## Local data

In [19]:
final_data = get_data(local_path)
print(len(final_data))

/projects/iris/rferreir/datasets/SpatialEvalLLM/map_local/type-hexagon_size-2_seed-2_n-10000_label-imagenetsimple_nsampled-100_nstep-8.jsonl
/projects/iris/rferreir/datasets/SpatialEvalLLM/map_local/type-ring_size-12_seed-12_n-10000_label-imagenetsimple_nsampled-100_nstep-8.jsonl
/projects/iris/rferreir/datasets/SpatialEvalLLM/map_local/type-square_size-3_seed-3_n-10000_label-imagenetsimple_nsampled-100_nstep-8.jsonl
/projects/iris/rferreir/datasets/SpatialEvalLLM/map_local/type-rhombus_size-3_seed-3_n-10000_label-imagenetsimple_nsampled-100_nstep-8.jsonl
/projects/iris/rferreir/datasets/SpatialEvalLLM/map_local/type-triangle_size-3_seed-3_n-10000_label-imagenetsimple_nsampled-100_nstep-8.jsonl
500


In [20]:
with open(os.path.join(output_dir, 'data_local.json'), 'w') as f:
    json.dump(final_data, f)

## Sanity Check

In [23]:
d1 = datasets.load_dataset('json', data_files={"test":[os.path.join(output_dir, 'data_global.json'), os.path.join(output_dir, 'data_local.json')]})
print(d1)

DatasetDict({
    test: Dataset({
        features: ['question', 'answer', 'scenario', 'struct_type', 'size', 'k_hop', 'seed', 'description_level'],
        num_rows: 1400
    })
})


In [25]:
d2 = datasets.load_dataset("rfr2003/Geo_Benchmark", "SpatialEvalLLM")

import hashlib
import json
# We check that the content of the dataset is the same
def content_hash(ds):
    h = hashlib.sha256()
    for ex in ds:
        h.update(json.dumps(ex, sort_keys=True).encode())
    return h.hexdigest()

assert content_hash(d1["test"]) == content_hash(d2["test"])

Generating test split: 100%|██████████| 1400/1400 [00:00<00:00, 72690.68 examples/s]
